In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import time
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# dataset path
dataset_path = "./data"

train_set = datasets.CIFAR10(root=dataset_path, train=True, download=True, transform=transform)
test_set  = datasets.CIFAR10(root=dataset_path, train=False, download=True, transform=transform)

# DataLoaders
train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

# ------------------ MODEL ------------------
model = models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

# AdamW optimizer
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

# Cosine LR scheduler
num_epochs = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# ------------------ TRAIN ------------------
print("Starting AdamW baseline training...")
start_total = time.time()

for epoch in range(num_epochs):
    model.train()
    start_epoch = time.time()

    correct, total = 0, 0

    for X, y in train_loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    scheduler.step()

    epoch_time = time.time() - start_epoch
    train_acc = 100 * correct / total
    print(f"Epoch {epoch+1}: Train Acc = {train_acc:.2f}%, Time = {epoch_time:.2f}s")

total_time = time.time() - start_total
print(f"Baseline completed. Total time: {total_time:.2f}s")

# ------------------ TEST ------------------
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        outputs = model(X)
        _, preds = torch.max(outputs, 1)
        correct += (preds == y).sum().item()
        total += y.size(0)

print(f"Test Accuracy: {100 * correct / total:.2f}%")


Using device: cuda


100%|██████████| 170M/170M [00:03<00:00, 48.2MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 143MB/s]


Starting AdamW baseline training...
Epoch 1: Train Acc = 69.98%, Time = 23.84s
Epoch 2: Train Acc = 81.46%, Time = 22.94s
Epoch 3: Train Acc = 86.42%, Time = 23.06s
Epoch 4: Train Acc = 89.54%, Time = 23.49s
Epoch 5: Train Acc = 92.36%, Time = 23.96s
Epoch 6: Train Acc = 94.50%, Time = 24.49s
Epoch 7: Train Acc = 95.98%, Time = 24.67s
Epoch 8: Train Acc = 96.95%, Time = 24.29s
Epoch 9: Train Acc = 97.60%, Time = 24.15s
Epoch 10: Train Acc = 98.37%, Time = 24.18s
Epoch 11: Train Acc = 98.82%, Time = 24.25s
Epoch 12: Train Acc = 99.27%, Time = 24.29s
Epoch 13: Train Acc = 99.54%, Time = 24.36s
Epoch 14: Train Acc = 99.68%, Time = 24.27s
Epoch 15: Train Acc = 99.74%, Time = 24.46s
Epoch 16: Train Acc = 99.88%, Time = 24.36s
Epoch 17: Train Acc = 99.93%, Time = 24.42s
Epoch 18: Train Acc = 99.95%, Time = 24.43s
Epoch 19: Train Acc = 99.98%, Time = 24.40s
Epoch 20: Train Acc = 99.98%, Time = 24.29s
Baseline completed. Total time: 482.60s
Test Accuracy: 84.95%
